# AnalogML — Technology Transfer Notebook
Transfer analog circuit design knowledge from 180nm → 90nm → 65nm

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from analogml.data   import SyntheticCircuitDataset
from analogml.models import AnalogMLModel, TechTransferAgent
from analogml.data.export import model_report, print_report

## 1. Multi-Technology Dataset

In [ ]:
techs = ['180nm','90nm','65nm']
data  = {}
for tech in techs:
    ds = SyntheticCircuitDataset(seed=hash(tech)%1000)
    X, Y, xn, yn = ds.generate('ota_5t', n_samples=250, technology=tech)
    data[tech] = (X, Y, xn, yn)
    print(f'{tech}: {X.shape}')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = {'180nm':'#2196F3','90nm':'#4CAF50','65nm':'#FF5722'}
for i, (tech, ax) in enumerate(zip(techs, axes)):
    X, Y, xn, yn = data[tech]
    ax.scatter(Y[:,0], Y[:,1], alpha=0.4, s=12, c=colors[tech])
    ax.set_xlabel('Gain (dB)')
    ax.set_ylabel('UGF (MHz)')
    ax.set_title(f'{tech} OTA')
plt.suptitle('Gain vs UGF across Technologies')
plt.tight_layout(); plt.show()

## 2. Train Source Model (180nm)

In [ ]:
X180, Y180, xn, yn = data['180nm']
perm = np.random.permutation(len(X180))
sp   = int(0.8*len(X180))

model_180 = AnalogMLModel(mode='fast', output_names=yn)
model_180.fit(X180[perm[:sp]], Y180[perm[:sp]], epochs=150)

rep = model_report(model_180, X180[perm[sp:]], Y180[perm[sp:]],
                    output_names=yn, path='/tmp/rep_180.json')
print_report(rep)

## 3. Technology Transfer 180nm → 90nm

In [ ]:
X90, Y90, _, _ = data['90nm']
n_common = min(len(X180), len(X90), 150)

agent = TechTransferAgent('180nm', '90nm')
agent.fit(X180[:n_common], X90[:n_common])

rules = agent.tech_scaling_rules()
print('Scaling rules:')
for k, v in rules.items():
    bar = '█' * max(1, int(abs(v)*8))
    print(f'  {k:<25} ×{v:.3f}  {bar}')

In [ ]:
# Evaluate transfer vs native
perm90 = np.random.permutation(len(X90))
X90_te = X90[perm90[int(0.8*len(X90)):]]
Y90_te = Y90[perm90[int(0.8*len(X90)):]]

X90_mapped  = agent.transfer(X90_te)
Y90_pred_transfer = model_180.predict(X90_mapped)

model_90 = AnalogMLModel(mode='fast', output_names=yn)
model_90.fit(X90[perm90[:int(0.8*len(X90))]], Y90[perm90[:int(0.8*len(X90))]])
Y90_pred_native = model_90.predict(X90_te)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, (name, ax) in enumerate(zip(yn[:3], axes)):
    ax.scatter(Y90_te[:,i], Y90_pred_transfer[:,i],
               alpha=0.5, s=15, c='#FF5722', label='Transfer')
    ax.scatter(Y90_te[:,i], Y90_pred_native[:,i],
               alpha=0.5, s=15, c='#2196F3', label='Native', marker='x')
    mn = Y90_te[:,i].min(); mx = Y90_te[:,i].max()
    ax.plot([mn,mx],[mn,mx],'k--',lw=1)
    ax.set_title(name, fontsize=9)
    ax.legend(fontsize=7)
plt.suptitle('90nm: Transfer Model vs Native Model')
plt.tight_layout(); plt.show()

## 4. Cascade Transfer: 180nm → 90nm → 65nm

In [ ]:
X65, Y65, _, _ = data['65nm']

# Two-hop transfer
agent_65 = TechTransferAgent('90nm','65nm')
n65 = min(len(X90), len(X65), 150)
agent_65.fit(X90[:n65], X65[:n65])

perm65   = np.random.permutation(len(X65))
X65_te   = X65[perm65[int(0.8*len(X65)):]]
Y65_te   = Y65[perm65[int(0.8*len(X65)):]]

# 180 → 90 → 65
X65_from90  = agent_65.transfer(agent.transfer(X65_te))
Y65_cascade = model_180.predict(X65_from90)

print('Cascade transfer (180→90→65) R² per output:')
for i, name in enumerate(yn):
    r2 = float(np.corrcoef(Y65_te[:,i], Y65_cascade[:,i])[0,1]**2)
    bar = '█' * int(r2*20)
    print(f'  {name:<30} R²={r2:.3f}  {bar}')